# Base vs SFT — Qwen2.5-1.5B Base + LoRA

В Kaggle включите **GPU T4 x2** и **Internet**. Ноутбук использует одну T4. В первой ячейке выберите режим и запустите **Run All**.

| `MODE` | Данные | Обучение |
| --- | --- | --- |
| `smoke` | Фикстуры из репозитория | 10 шагов, проверка пайплайна |
| `probe` | 8 222 train и 300 val | 50 шагов, validation и checkpoint каждые 25 |
| `full` | Те же train и val | 2 эпохи, validation и checkpoint каждые 50 |

После probe проверьте loss, grad norm, скорость и видеопамять. Затем выберите `full` и запустите **Save Version → Save & Run All**. Full стартует с Base. Адаптер с минимальным validation loss, метрики, лог и график попадут в ZIP в **Output**.

In [ ]:
MODE = "probe"  # "smoke", "probe", "full"
TRAIN_BATCH_SIZE = 2  # Для probe/full можно выбрать 1; эффективный batch останется 16.
RUN_SUFFIX = ""  # Для повторного запуска в той же сессии, например "retry1".
SPIDER_INPUT = None  # Или путь к папке с raw Spider и database/ в /kaggle/input.

## Код, GPU и зависимости

Используем версию командного пайплайна, зафиксированную по коммиту. В ноутбук включена копия YAML, чтобы импортированный `.ipynb` можно было запустить сразу.

In [ ]:
import json
import os
import re
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

if MODE not in {"smoke", "probe", "full"}:
    raise ValueError("MODE должен быть smoke, probe или full.")
if TRAIN_BATCH_SIZE not in {1, 2}:
    raise ValueError("TRAIN_BATCH_SIZE должен быть 1 или 2.")
if RUN_SUFFIX and not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_SUFFIX):
    raise ValueError("RUN_SUFFIX может содержать только буквы, цифры, _ и -.")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run(["nvidia-smi", "-L"], check=True)

WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "BaseModel-vs-SFT-1.5b"
REPO_URL = "https://github.com/Pavel-V418/BaseModel-vs-SFT.git"
REPO_COMMIT = "9b02b741ae880794c09f48dc76ddd609aa5b82cc"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_COMMIT], check=True)
os.chdir(REPO_DIR)
print("Код проекта", REPO_COMMIT)

# Эти версии записаны в метриках успешного прогона 0.5B на 8 222 примерах.
with TemporaryDirectory() as tmp:
    constraints = Path(tmp) / "constraints.txt"
    constraints.write_text(f"torch=={version('torch')}\n", encoding="utf-8")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints),
         "transformers==5.20.2", "trl==1.18.1", "peft==0.21.3", "datasets==4.8.5",
         "accelerate", "pyyaml", "matplotlib", "gdown"],
        check=True,
    )
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"], check=True)
subprocess.run(
    [sys.executable, "-c",
     "import torch, transformers, trl, peft, datasets; "
     "assert torch.cuda.device_count() == 1, 'Ожидается одна видимая CUDA GPU'; "
     "print(torch.cuda.get_device_name(0), torch.__version__)"],
    check=True,
)

## Конфиг и режим

Smoke использует штатный `--smoke` со своими batch size и gradient accumulation. Probe и full используют одинаковый batch. При CUDA OOM в probe установите `TRAIN_BATCH_SIZE = 1`, задайте новый `RUN_SUFFIX` или начните новую сессию и повторите probe. Для full сохраните проверенный batch.

In [ ]:
from src.utils import load_config

CONFIG = Path("configs/sft_1.5b.yaml")
CONFIG_TEXT = """\
# Qwen2.5-1.5B Base + LoRA на расширенном Spider.
# Подготовка: python -m src.data --spider_dir data/spider --out_dir data/spider_others --val_size 400 --seed 42 --include_others
run_name: sft_1.5b_nfull_others_r16_lr1e-4_s42_2ep
model: Qwen/Qwen2.5-1.5B
method: lora
seed: 42

data:
  train: data/spider_others/train.jsonl
  val: data/spider_others/val.jsonl
train_size: null
val_size: 300
completion_only_loss: true

lora:
  r: 16
  alpha: 32
  dropout: 0.05
  target_modules: all-linear

training:
  learning_rate: 1.0e-4
  num_train_epochs: 2
  per_device_train_batch_size: 2
  per_device_eval_batch_size: 2
  gradient_accumulation_steps: 8
  max_length: 1024
  warmup_steps: 50
  lr_scheduler_type: cosine
  eval_steps: 50
  logging_steps: 10
  gradient_checkpointing: true

# src.train выбирает fp16 на T4 и bf16 на Ampere и новее.
report_to: none
hub_repo: null
"""
if not CONFIG.exists():
    CONFIG.write_text(CONFIG_TEXT, encoding="utf-8", newline="\n")
cfg = load_config(CONFIG)
base_name = cfg["run_name"]
if MODE == "probe":
    base_name = f"probe_{base_name}"

OVERRIDES = []
if RUN_SUFFIX:
    base_name = f"{base_name}_{RUN_SUFFIX}"
OVERRIDES.append(f"run_name={base_name}")
if MODE != "smoke":
    OVERRIDES.extend([
        f"training.per_device_train_batch_size={TRAIN_BATCH_SIZE}",
        f"training.per_device_eval_batch_size={TRAIN_BATCH_SIZE}",
        f"training.gradient_accumulation_steps={16 // TRAIN_BATCH_SIZE}",
    ])
if MODE == "probe":
    OVERRIDES.extend([
        "training.max_steps=50",
        "training.eval_steps=25",
        "training.warmup_steps=5",
        "training.logging_steps=5",
    ])

RUN_NAME = f"smoke_{base_name}" if MODE == "smoke" else base_name
LIVE_PATH = Path("results") / f"{RUN_NAME}_live.json"
LOG_PATH = Path("results") / f"{RUN_NAME}_stdout.log"
OUTPUT_DIR = Path("outputs") / RUN_NAME
ARCHIVE = WORK_DIR / f"results_{RUN_NAME}.zip"
if OUTPUT_DIR.exists() or LOG_PATH.exists() or ARCHIVE.exists():
    raise FileExistsError(f"{RUN_NAME} уже запускался. Задайте новый RUN_SUFFIX или начните новую сессию.")
print(RUN_NAME)

## Расширенный Spider

В probe и full используем штатный `src.data --include_others` с тем же разбиением по базам и seed 42. `src.train` отбирает 300 validation-примеров. Smoke использует только тестовые фикстуры.

In [ ]:
import shutil
from contextlib import closing
import sqlite3

if MODE != "smoke":
    spider_dir = Path(SPIDER_INPUT) if SPIDER_INPUT else Path("data/spider")
    if SPIDER_INPUT is None:
        # Скрипт завершает работу проверкой через sqlite3 CLI, которого может не быть.
        subprocess.run(
            ["bash", "scripts/download_spider.sh", str(spider_dir)],
            check=shutil.which("sqlite3") is not None,
        )

    required = ("train_spider.json", "train_others.json", "dev.json", "tables.json")
    missing = [name for name in required if not (spider_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"В {spider_dir} отсутствуют файлы Spider: {missing}")
    sample_db = (spider_dir / "database/concert_singer/concert_singer.sqlite").resolve()
    with closing(sqlite3.connect(sample_db.as_uri() + "?mode=ro", uri=True)) as conn:
        conn.execute("SELECT count(*) FROM singer").fetchone()

    data_dir = Path(cfg["data"]["train"]).parent
    subprocess.run(
        [sys.executable, "-m", "src.data", "--spider_dir", str(spider_dir),
         "--out_dir", str(data_dir), "--val_size", "400", "--seed", str(cfg["seed"]),
         "--include_others", "--show", "0"],
        check=True,
    )
    stats = json.loads((data_dir / "stats.json").read_text(encoding="utf-8"))["splits"]
    counts = {name: stats[name]["n_examples"] for name in ("train", "val", "test")}
    print(counts)
    if counts["train"] != 8222 or counts["val"] < cfg["val_size"]:
        raise RuntimeError(f"Ожидались 8 222 train и минимум {cfg['val_size']} val, получено {counts}.")

## Запуск обучения и мониторинг

Для просмотра полного лога откройте `results/<run>_stdout.log`. Прерывание ячейки мониторинга оставляет процесс обучения в фоне. Чтобы продолжить наблюдение, запустите ячейку мониторинга ещё раз.

In [ ]:
cmd = [sys.executable, "-u", "-m", "src.train", "--config", str(CONFIG)]
if MODE == "smoke":
    cmd.append("--smoke")
for override in OVERRIDES:
    cmd.extend(["--set", override])

LOG_PATH.parent.mkdir(exist_ok=True)
with LOG_PATH.open("x", encoding="utf-8") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print(f"{RUN_NAME} | PID {proc.pid} | {LOG_PATH}")

In [ ]:
import time
from collections import deque

import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from src.plots import plot_history

peak_sampled_mib = 0
while True:
    clear_output(wait=True)
    code = proc.poll()
    print(RUN_NAME, "|", "обучение идёт" if code is None else f"код выхода {code}")
    gpu = subprocess.run(
        ["nvidia-smi", "-i", "0", "--query-gpu=memory.used,memory.total",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    if gpu.returncode == 0:
        used, total = (int(value) for value in gpu.stdout.strip().split(","))
        peak_sampled_mib = max(peak_sampled_mib, used)
        print(f"VRAM — {used} / {total} МиБ | максимум по замерам {peak_sampled_mib} МиБ")
    with LOG_PATH.open(encoding="utf-8", errors="replace") as log:
        print("".join(deque((line for line in log if line.strip()), maxlen=12)))
    if LIVE_PATH.exists():
        fig = plot_history(LIVE_PATH)
        display(fig)
        plt.close(fig)
    if code is not None:
        if code:
            raise RuntimeError(f"Обучение завершилось с кодом {code}. Полный лог: {LOG_PATH}")
        break
    time.sleep(15)

## Результат и архив

Архив текущего запуска включает итоговый LoRA-адаптер из `final/`, JSON-метрики, лог и график. В полном запуске `final/` содержит веса чекпоинта с минимальным validation loss.

In [ ]:
import math
from zipfile import ZIP_DEFLATED, ZipFile

from IPython.display import FileLink, Image

summary_path = Path("results") / f"{RUN_NAME}_train.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
live = json.loads(LIVE_PATH.read_text(encoding="utf-8"))
expected_steps = {"smoke": 10, "probe": 50}.get(MODE)
if live["phase"] != "done":
    raise RuntimeError("Обучение не дошло до финального сохранения.")
if expected_steps is not None and live["max_steps"] != expected_steps:
    raise RuntimeError(f"Ожидалось {expected_steps} шагов, получено {live['max_steps']}.")
if MODE != "smoke" and (summary["train_size"], summary["val_size"]) != (8222, 300):
    raise RuntimeError("Размер загруженных train/val не совпадает с настройками эксперимента.")

before = summary["eval_before"]["eval_loss"]
after = summary["eval_after"]["eval_loss"]
train_log = [row for row in live["history"] if row["phase"] == "train" and "loss" in row]
if (not train_log or not all(math.isfinite(x) for x in (before, after))
        or any(not math.isfinite(row["loss"]) or not math.isfinite(row.get("grad_norm", 0))
               for row in train_log)):
    raise RuntimeError(f"Проверьте loss и grad_norm в {LOG_PATH}.")
final_dir = OUTPUT_DIR / "final"
if not summary["best_checkpoint"] or not (final_dir / "adapter_model.safetensors").is_file():
    raise RuntimeError("Не найден лучший checkpoint или файл весов адаптера.")

print(f"Train / val — {summary['train_size']} / {summary['val_size']}")
print(f"Validation loss до / после SFT — {before:.4f} / {after:.4f}")
print(f"Время обучения — {summary['train_time_sec'] / 60:.1f} мин")
print("Лучший checkpoint", summary["best_checkpoint"])
print("Максимум VRAM по замерам", peak_sampled_mib, "МиБ")

png = Path("results") / f"{RUN_NAME}_curves.png"
if not png.exists():
    plt.close(plot_history(LIVE_PATH, out=png))
display(Image(filename=str(png)))

artifacts = [summary_path, LIVE_PATH, LOG_PATH, png,
             Path("results") / f"{RUN_NAME}_log_history.json"]
if MODE != "smoke":
    artifacts.append(data_dir / "stats.json")
artifacts.extend(sorted(path for path in final_dir.rglob("*") if path.is_file()))
with ZipFile(ARCHIVE, "x", ZIP_DEFLATED) as bundle:
    for path in artifacts:
        bundle.write(path, arcname=path.as_posix())

os.chdir(WORK_DIR)
try:
    display(FileLink(ARCHIVE.name))
finally:
    os.chdir(REPO_DIR)